# Mean Shift — Solutions Notebook

**Difficulty**: ⭐⭐⭐ Advanced  
**Time**: ~45 mins  
**Complete, verified reference implementation.**

---


## 🎯 Section 1: Overview

A centroid-based clustering algorithm that shifts points towards density modes (modes of kernel density estimation).

### Mean Shift Vector:
$$m(x) = \frac{\sum_{i} x_i K(\frac{x_i - x}{h})}{\sum_i K(\frac{x_i - x}{h})} - x$$


## 🔧 Section 2: Implementation from Scratch


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

np.random.seed(42)
plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('husl')

print('Setup complete! ✅')

In [ ]:
class MeanShiftFromScratch:
    def __init__(self, bandwidth=1.0, tolerance=1e-3, max_iters=100):
        self.bandwidth = bandwidth
        self.tolerance = tolerance
        self.max_iters = max_iters
        
    def fit(self, X):
        centroids = np.copy(X)
        for _ in range(self.max_iters):
            new_centroids = []
            for c in centroids:
                # Gaussian kernel weights
                distances = np.linalg.norm(X - c, axis=1)
                weights = np.exp(-0.5 * (distances / self.bandwidth) ** 2)
                weighted_mean = np.sum(X * weights[:, np.newaxis], axis=0) / np.sum(weights)
                new_centroids.append(weighted_mean)
            
            new_centroids = np.array(new_centroids)
            shift = np.linalg.norm(new_centroids - centroids)
            if shift < self.tolerance:
                break
            centroids = new_centroids
            
        # Group close centroids
        unique_centroids = []
        for c in centroids:
            if not any(np.linalg.norm(c - uc) < self.bandwidth * 0.5 for uc in unique_centroids):
                unique_centroids.append(c)
                
        self.cluster_centers_ = np.array(unique_centroids)
        # Assign labels
        distances_to_centers = np.linalg.norm(X[:, np.newaxis] - self.cluster_centers_, axis=2)
        self.labels_ = np.argmin(distances_to_centers, axis=1)
        return self


In [ ]:
# Verify that the implementation runs and outputs correctly
X = np.random.rand(50, 2)
model = MeanShiftFromScratch(bandwidth=0.3)
model.fit(X)
print('Centers found:', len(model.cluster_centers_))


## 📦 Section 3: Library Implementation


In [ ]:
from sklearn.cluster import MeanShift
ms = MeanShift(bandwidth=1.5)
labels = ms.fit_predict(X)


## ❓ Section 4: Interview Questions


### Q1: Does Mean Shift require pre-defining the cluster count K?
**Answer**: No, it dynamically finds density peaks/clusters. However, the number of clusters depends heavily on the bandwidth parameter.


### Q2: What is the physical interpretation of the bandwidth parameter?
**Answer**: It defines the search window size for kernel density estimation. A small bandwidth yields many small clusters; a large one merges clusters.
